In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from pathlib import Path

user_secrets = UserSecretsClient()

import time


def get_secret_with_retry(label, retries=5, delay=3):
    # Kaggle Secrets サービスが一時的に 400/ConnectionError を返すことがあるためリトライする
    for attempt in range(retries):
        try:
            return user_secrets.get_secret(label)
        except Exception as e:
            print(f"get_secret('{label}') 失敗 (試行 {attempt + 1}/{retries}): {e}")
            if attempt == retries - 1:
                raise
            time.sleep(delay)


import json

# Secrets は private Kaggle Dataset (voice-clone-secrets) から読む。
# API push (kernels_push) では手動アタッチした Kaggle Secrets が引き継がれないため、
# データセット経由で渡す。データセットが無ければ get_secret にフォールバックする。
_SECRETS_PATH = Path("/kaggle/input/voice-clone-secrets/secrets.json")
_secrets = json.loads(_SECRETS_PATH.read_text()) if _SECRETS_PATH.exists() else None


def secret(label):
    if _secrets is not None and label in _secrets:
        return _secrets[label]
    return get_secret_with_retry(label)


BASE_URL = "https://neglegere.com"
EMAIL = secret("email")
PASSWORD = secret("password")
SAVE_DIR = Path("/kaggle/working/reference_audio")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_TOKEN"] = secret("hf_token")

R2_ACCOUNT_ID = secret("r2_AccountID")
R2_ACCESS_KEY = secret("r2_accesskey")
R2_SECRET_KEY = secret("r2_secretkey")
R2_BUCKET_NAME = secret("r2_bucket_name")
R2_ENDPOINT = f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com"

WEBHOOK_URL = secret("webhook_url")


def notify_slack(text: str):
    import requests
    try:
        requests.post(WEBHOOK_URL, json={"text": text}, timeout=15)
    except Exception as e:
        print("Slack通知に失敗:", e)


# --- pending を先に確認し、0件なら以降の重い処理をスキップする ---
import boto3
import json as _json


def fetch_pending_voice_clones(key: str = "pending.json"):
    s3 = boto3.client(
        "s3",
        endpoint_url=R2_ENDPOINT,
        aws_access_key_id=R2_ACCESS_KEY,
        aws_secret_access_key=R2_SECRET_KEY,
    )
    res = s3.get_object(Bucket=R2_BUCKET_NAME, Key=key)
    return _json.loads(res["Body"].read().decode("utf-8"))


_pending = fetch_pending_voice_clones()
HAS_WORK = bool(_pending and _pending.get("items"))
print(f"pending: {len(_pending.get('items', [])) if _pending else 0} 件")
if not HAS_WORK:
    print("0件のため pip install / モデル処理をスキップします")


In [ ]:
# torch/soundfile/transformers/accelerate は Kaggle に同梱済みのため入れない
# pending がある時だけ qwen-tts を入れる（0件なら pip を走らせない）
if HAS_WORK:
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "qwen-tts", "librosa", "scipy"],
        check=True,
    )

import os
import torch
import soundfile as sf
from pathlib import Path


import signal


class GenerationTimeout(Exception):
    pass


_qwen_model_cache = {}


def _get_qwen_model(model_name):
    # モデルはプロセス内で1回だけロードして使い回す（item ごとの再ロードを避ける）
    if model_name not in _qwen_model_cache:
        from qwen_tts import Qwen3TTSModel

        device = "cuda" if torch.cuda.is_available() else "cpu"
        _qwen_model_cache[model_name] = Qwen3TTSModel.from_pretrained(
            model_name,
            device_map="cuda:0" if device == "cuda" else "cpu",
            dtype=torch.float16 if device == "cuda" else torch.float32,
        )
        print(f"✅ Model loaded on {device}")
    return _qwen_model_cache[model_name]


def generate_qwen_voice(
    text: str,
    ref_audio_path: str,
    ref_text: str,
    output_path: str = "/kaggle/working/qwen_generated.wav",
    model_name: str = "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
):
    """
    Qwen3-TTSを使用してボイスクローン音声を生成するメソッド。

    Args:
        text (str): 生成する音声のテキスト
        ref_audio_path (str): 参照音声（WAVファイル）のパス
        ref_text (str): 参照音声の読み上げ内容
        output_path (str): 生成された音声の保存先
        model_name (str): 使用するQwenモデル

    Returns:
        str: 生成された音声ファイルのパス
    """

    # モデルは1回だけロードして使い回す
    model = _get_qwen_model(model_name)

    # 音声生成（10分でタイムアウトして中断）
    def _on_generate_timeout(signum, frame):
        raise GenerationTimeout("generate_voice_clone が10分を超えたため中断しました")

    old_handler = signal.signal(signal.SIGALRM, _on_generate_timeout)
    signal.alarm(600)
    try:
        wavs, sr = model.generate_voice_clone(
            text=text,
            language="Japanese",
            ref_audio=ref_audio_path,
            ref_text=ref_text,
        )
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)

    # 保存先のディレクトリを作成
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # WAVファイルとして保存
    sf.write(str(output_path), wavs[0], sr)

    print(f"✅ Generated voice saved: {output_path}")

    return str(output_path)

In [ ]:
import requests
from pathlib import Path

def login() -> str:
    url = f"{BASE_URL}/api/auth/login"
    headers = {
        "accept": "application/json",
        "Content-Type": "application/json",
    }
    payload = {
        "email": EMAIL,
        "password": PASSWORD,
    }

    res = requests.post(url, headers=headers, json=payload, timeout=30)
    res.raise_for_status()

    data = res.json()

    token = data.get("token")

    return token

def get_pending_items(token: str) -> list[dict]:
    url = f"{BASE_URL}/api/voice_clones/pending"
    headers = {
        "accept": "application/json",
        "Authorization": f"Bearer {token}",
    }

    res = requests.get(url, headers=headers, timeout=30)
    res.raise_for_status()

    data = res.json()

    items = data.get("items", [])

    return items

def download_reference_audio(token: str, voice_clone_id: str, file_name: str | None = None) -> Path:
    url = f"{BASE_URL}/api/reference_audio/{voice_clone_id}/file"
    headers = {
        "accept": "audio/wav",
        "Authorization": f"Bearer {token}",
    }

    res = requests.post(url, headers=headers, data="", stream=True, timeout=60)
    res.raise_for_status()

    output_name = Path(file_name).name

    output_path = SAVE_DIR / output_name

    with open(output_path, "wb") as f:
        for chunk in res.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

    return output_path

def upload_completed_voice_clone(
    token: str,
    voice_clone_id: str,
    audio_file_path,
):
    audio_file_path = Path(audio_file_path)

    url = f"{BASE_URL}/api/voice_clones/{voice_clone_id}/complete"

    headers = {
        "accept": "application/json",
        "Authorization": f"Bearer {token}",
    }

    with open(audio_file_path, "rb") as f:
        files = {
            "file": (audio_file_path.name, f, "audio/wav")
        }

        res = requests.post(
            url,
            headers=headers,
            files=files,
            timeout=60,
        )

    res.raise_for_status()

    data = res.json()

    if not data.get("ok"):
        raise Exception(f"APIエラー: {data}")

    return data


def fail_voice_clone(token: str, voice_clone_id: str):
    url = f"{BASE_URL}/api/voice_clones/{voice_clone_id}/fail"

    headers = {
        "accept": "application/json",
        "Authorization": f"Bearer {token}",
    }

    res = requests.post(url, headers=headers, timeout=30)
    res.raise_for_status()

    return res.json()


In [ ]:
import boto3
import json

def fetch_pending_voice_clones(key: str = "pending.json"):
    s3 = boto3.client(
        "s3",
        endpoint_url=R2_ENDPOINT,
        aws_access_key_id=R2_ACCESS_KEY,
        aws_secret_access_key=R2_SECRET_KEY,
    )

    res = s3.get_object(
        Bucket=R2_BUCKET_NAME,
        Key=key,
    )

    return json.loads(res["Body"].read().decode("utf-8"))

print(fetch_pending_voice_clones())

In [ ]:
import shutil
import time

def main():

    while True:
        items = fetch_pending_voice_clones()

        if not items or not items.get("items"):
            print("pending は 0 件です")
            break

        for item in items["items"]:
            Path("/kaggle/working/reference_audio").mkdir(parents=True, exist_ok=True)
            Path("/kaggle/working/clone").mkdir(parents=True, exist_ok=True)
    
            voice_clone_id = item["id"]
            referenceAudioPath = item["referenceAudioPath"]

            print(f"downloading id={voice_clone_id} ...")
            token = login()
            output_path = download_reference_audio(
                token, 
                voice_clone_id, 
                referenceAudioPath
            )
            print(f"saved: {output_path}")

            # 音声生成（10分超はスキップ）
            try:
                output_file = generate_qwen_voice(
                    text=item["desiredText"],
                    ref_audio_path=str(output_path),
                    ref_text=item["recordedText"],
                    output_path=Path("/kaggle/working/clone/") / Path(referenceAudioPath).name,
                )
            except GenerationTimeout as e:
                print(f"skip id={voice_clone_id}: {e}")
                notify_slack(
                    f":warning: voice-clone-task-by-T4: id={voice_clone_id} が10分を超えたためスキップしました"
                )
                try:
                    fail_voice_clone(login(), voice_clone_id)
                except Exception as fe:
                    print(f"fail 通知に失敗: {fe}")
                shutil.rmtree("/kaggle/working/reference_audio", ignore_errors=True)
                shutil.rmtree("/kaggle/working/clone", ignore_errors=True)
                continue

            upload_completed_voice_clone(
                token,
                voice_clone_id,
                output_file
            )
        
            print("生成されたファイル:", output_file)
        
            shutil.rmtree("/kaggle/working/reference_audio")
            shutil.rmtree("/kaggle/working/clone")

    print("\n完了")

import traceback

if HAS_WORK:
    try:
        main()
    except Exception:
        tb = traceback.format_exc()
        print(tb)
        notify_slack(
            f":rotating_light: voice-clone-task-by-T4 が失敗しました\n```\n{tb[-3000:]}\n```"
        )
        raise
else:
    print("pending 0件のため main() を実行しません")
